# Test Model wih ROI Data

Model: `(16,16,2)`

## List trained models

In [ ]:
!for f in `ls weights/dataset_3src_16x16_weights/*/*.json`; do echo $f; done

## Setup

Please follow [these instructions](https://github.com/GiuseppeDiGuglielmo/catapult_venvs) to setup an environment for Catapult AI NN (_hls4ml_).

You can either use the _hls4ml_ release with Catapult (2024.2_1) or point to a Siemens or official GitHub repository. [Here we provide some details.](https://github.com/GiuseppeDiGuglielmo/catapult_compare_releases)

In [ ]:
# Path to the Catapult installation directory
!echo $MGC_HOME

In [ ]:
# Path to the hls4ml installation directory
# It can be point to either the Catapult installation directory or a copy of Siemens/Official GitHub repo 
!echo $PYTHONPATH

## Import libraries

Disable some console warnings on the ASIC-group servers

In [ ]:
import os

In [ ]:
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [ ]:
import hls4ml

import numpy as np
import math
import yaml
import json
from models.mlp_encoder_model import *

from matplotlib import colors
import matplotlib.pyplot as plt

from loss import *
from qkeras import quantized_bits

## Set parameters

In [ ]:
#dataset_base_dir = "/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/"
dataset_base_dir = "/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/"

tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")
npy_base_dir = os.path.join(dataset_base_dir, "NPY_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train_contained_digitize-manual_mlp-SLIM")
dataset_validation_dir = os.path.join(dataset_base_dir, "test_contained_digitize-manual_mlp-SLIM")

npy_dir_val = os.path.join("npy")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train_contained_digitize-manual_mlp-SLIM")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val_contained_digitize-manual_mlp-SLIM")

In [ ]:
batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_validation_dir))

In [ ]:
base_dir = 'weights/dataset_3src_16x16_weights/'
!ls $base_dir

pitch = '50x12P5'
fingerprint = '34c2da80'

In [ ]:
weights_dir = base_dir + 'weights-{}-bs{}-{}-2t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint)
best_model_hdf5 = f"{weights_dir}/best_model-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.hdf5"
best_model_keras = f"{weights_dir}/best_model-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.keras"
best_model_weights_hdf5 = f"{weights_dir}/best_model_weights-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.hdf5"
best_model_weights_keras = f"{weights_dir}/best_model_weights-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.keras"
model_architecture_json = f"{weights_dir}/model_architecture-{pitch}-bs{batch_size}-{fingerprint}-2t-mlp_SLIM-model_quantized.json"

In [ ]:
print(best_model_hdf5)

## Load data

In [ ]:
ROI_DATA_ENABLED = True

if ROI_DATA_ENABLED:
    X = np.load(f'/home/giuseppe/research/projects/smartpixels/VIAS/find_charge_cluster_centers/tb/cocotb/cluster_id/dataset_2su_-9_9_48x192_50x12P5_npy/test/X_part.0.npy')
    X = np.moveaxis(X.reshape((X.shape[0], 2, 16, 16)), 1, -1)
    y = np.load(f'/home/giuseppe/research/projects/smartpixels/VIAS/find_charge_cluster_centers/tb/cocotb/cluster_id/dataset_2su_-9_9_48x192_50x12P5_npy/test/y_part.0.npy')
else:    
    X = np.load(f'{npy_dir_val}/X_val.npy')
    y = np.load(f'{npy_dir_val}/y_val.npy')

In [ ]:
print(X.shape)
print(y.shape)

### Visualize data

Data visualization can also be moved to a different notebook, which may also already exists.

In [ ]:
def plot_event(X, timeslices_range, index):
    if index < 0 or index >= X.shape[0]:
        print("Index out of range. Please provide a valid index.")
        return
    
    timeframe_num = X[index].shape[2]
    
    fig, axes = plt.subplots(1, timeframe_num, figsize=(timeframe_num*4, 16))

    vmin = float(X[index].min())
    vmax = float(X[index].max())
    divnorm = colors.TwoSlopeNorm(vmin=vmin, vcenter=(vmin+vmax)/2, vmax=vmax)

    imgs = []
    for i_slice in range(timeframe_num):
        img = axes[i_slice].imshow(
            X[index, :, :, i_slice],
            interpolation='nearest',
            origin='lower',
            cmap='bwr',
            norm=divnorm
        )
        imgs.append(img)
        axes[i_slice].set_title(f'Event: {index} - Timeframe {timeslices_range[i_slice]}')
        axes[i_slice].set_xticks(range(0, X.shape[2], max(1, X.shape[2] // 5)))
        axes[i_slice].set_yticks(range(0, X.shape[1], max(1, X.shape[1] // 5)))

    # shared colorbar on the right, not symmetric but including both negative and positive ticks if present
    cbar = fig.colorbar(
        imgs[0], ax=axes.ravel().tolist(), orientation='vertical',
        fraction=0.025, pad=0.08, aspect=20
    )
    cbar.set_label('Value')

    # ensure both negative and positive ticks appear when data spans zero
    if vmin < 0 and vmax > 0:
        import numpy as np
        ticks = np.concatenate([np.linspace(vmin, 0.0, 3, endpoint=False),
                                np.linspace(0.0, vmax, 3)])
        cbar.set_ticks(ticks.tolist())

    plt.show()

# Plot some images
for i in range(5):
    plot_event(X, range(2), i)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_event(X, timeslices_range, index):
    if index < 0 or index >= X.shape[0]:
        print("Index out of range. Please provide a valid index.")
        return
    
    timeframe_num = X[index].shape[2]
    fig, ax = plt.subplots(figsize=(6, 6))

    vmin = float(X[index].min())
    vmax = float(X[index].max())
    divnorm = colors.TwoSlopeNorm(vmin=vmin, vcenter=(vmin + vmax)/2, vmax=vmax)

    img = ax.imshow(
        X[index, :, :, 0],
        interpolation='nearest',
        origin='lower',
        cmap='bwr',
        norm=divnorm
    )

    ax.set_title(f'Event: {index} - Timeframe 0')
    ax.set_xticks(range(0, X.shape[2], max(1, X.shape[2] // 5)))
    ax.set_yticks(range(0, X.shape[1], max(1, X.shape[1] // 5)))

    # thin colorbar on the right; include both negative and positive ticks if data spans zero
    cbar = fig.colorbar(img, ax=ax, orientation='vertical', fraction=0.025, pad=0.02, aspect=40)
    cbar.set_label('Value')
    if vmin < 0 and vmax > 0:
        import numpy as np
        ticks = np.concatenate([
            np.linspace(vmin, 0.0, 3, endpoint=False),
            np.linspace(0.0, vmax, 3)
        ])
        cbar.set_ticks(ticks.tolist())

    def update(frame):
        img.set_data(X[index, :, :, frame])
        ax.set_title(f'Event: {index} - Timeframe {timeslices_range[frame]}')
        return img, ax

    ani = FuncAnimation(fig, update, frames=timeframe_num, blit=False, interval=500)

    plt.close(fig)
    return ani


ani = animate_event(X, range(2), 2)
#ani.save("animation.mp4", fps=2)
ani.save(f"animation.gif", writer='imagemagick')

HTML(ani.to_jshtml())

### Quantize data

<b style="background-color: yellow; color: red">ATTENTION: Input quantization is disabled. The _hls4ml_ implementation uses the default value: <i style="font-family: Courier;">fixed<16,6></i></b>

## QKeras model

### Load QKeras model

In [ ]:
# Load the whole model from HDF5 file
from tensorflow.keras.models import load_model
from qkeras.utils import _add_supported_quantized_objects

co = {"custom_sse_loss": custom_sse_loss}
_add_supported_quantized_objects(co)
model = load_model(best_model_hdf5, custom_objects=co)
model.summary(line_length=120, show_trainable=True)

In [ ]:
# TODO: This gives an error:
# ValueError: Layer count mismatch when loading weights from file. Model expected 5 layers, found 9 saved layers.
#
# # Load model weights from HDF5 file, while recreate architecture from scripts
# model = CreateModel((13,21,2), n_filters=5, pool_size=3)
# model.load_weights(best_model_weights_hdf5)
# model.summary(line_length=200)

### Slice QKeras model

<b style="background-color: yellow; color: red">ATTENTION: You can select a subset of layers of the model (slice) to debug the HLS implementation.</b>

In [ ]:
sliced_model_enable = False
slice_from_layer_num = 6
slice_to_layer_num = 6
slice_away_last_activation = True

In [ ]:
# for l in model.layers:
#     print(l, l.input)
#     print(l, l.output)

In [ ]:
if sliced_model_enable:
    print("Layer count: {}/{}".format(slice_to_layer_num, len(model.layers)-1))
    assert(slice_to_layer_num < len(model.layers))
    model_head = Model(inputs=model.input,
                  outputs=model.layers[slice_to_layer_num-1].output)

    model_head.summary(line_length=120, show_trainable=True)

    from tensorflow.keras.utils import plot_model
    plot_model(model_head, to_file=f"{model_name}_model_head.png", show_shapes=True, show_layer_names=True)


if sliced_model_enable:

    assert slice_from_layer_num < len(model.layers), f"slice_from_layer_num ({slice_from_layer_num}) exceeds layer count ({len(model.layers)})"
    assert slice_to_layer_num < len(model.layers), f"slice_to_layer_num ({slice_to_layer_num}) exceeds layer count ({len(model.layers)})"
    assert slice_from_layer_num <= slice_to_layer_num, "slice_from_layer_num must be less than or equal to slice_to_layer_num"

    print("Layer count: {}/{}".format(slice_to_layer_num, len(model.layers) - 1))

    # Create new input tensor based on the shape of the selected layer
    input_shape = model.layers[slice_from_layer_num].input_shape[1:]
    new_input = tf.keras.layers.Input(shape=input_shape)

    # Connect the new input to the desired layers
    x = model.layers[slice_from_layer_num](new_input)
    for i in range(slice_from_layer_num + 1, slice_to_layer_num + 1):
        x = model.layers[i](x)

    # Create a new sliced model
    model = Model(inputs=new_input, outputs=x)
    model.summary(line_length=120, show_trainable=True)

    from tensorflow.keras.utils import plot_model
    plot_model(model, to_file=f"{model_name}_model.png", show_shapes=True, show_layer_names=True)

### Run QKeras model

In [ ]:
print(f'Input batch: {X.shape[0]}')
print(f'Input shape: {X.shape}')

#### Prediction

In [ ]:
if sliced_model_enable:
    y_head_qkeras = model_head.predict(np.ascontiguousarray(X))
    y_qkeras = model.predict(y_head_qkeras)
else:
    y_qkeras = model.predict(np.ascontiguousarray(X))
print(f'QKeras output batch: {y_qkeras.shape[0]}')
print(f'QKeras output shape: {y_qkeras.shape}')

#### Profiling

In [ ]:
trace_qkeras = hls4ml.model.profiling.get_ymodel_keras(model, X)
#print(trace_qkeras['q_separable_conv2d'].shape)

In [ ]:
for key in trace_qkeras.keys():
    print(f'QKeras layer trace shape: {key} {trace_qkeras[key].shape}')

#### Save .dat files

In [ ]:
# Save input features and model predictions just the top 20
np.savetxt(f"tb_input_features.dat", X.reshape(X.shape[0], -1), fmt="%f")
np.savetxt(f"tb_output_predictions.dat", y_qkeras.reshape(X.shape[0], -1), fmt="%f")
#np.savetxt("y_test_labels.dat", y_test, fmt="%d")

In [ ]:
!wc -l *tb_input_features.dat
!wc -l *tb_output_predictions.dat

In [ ]:
if ROI_DATA_ENABLED:    
    roi_scale_factor = {
        'x':99.1906681423022,
        'y':24.80357256924443,
        'cotA':6.536435788653136,
        'cotB':1.885160054237276
    }
    print('reference\n', y[:10, 21:])
    print('predicted\n', y_qkeras[:10, :2])
    print('predicted scaled\n', y_qkeras[:10, :2] * [roi_scale_factor['x'], roi_scale_factor['y']])
else:
    print('reference\n', y[:10,:2])
    print('predicted\n', y_qkeras[:10, :2])

In [ ]:
def mse(actual, predicted):
    return ((actual - predicted) ** 2).mean()

N = 5000

In [ ]:
if ROI_DATA_ENABLED:
    print(f"MSE(x) {mse(y[:N, 21], y_qkeras[:N, 0] * roi_scale_factor['x'])}")
    print(f"MSE(y) {mse(y[:N, 12], y_qkeras[:N, 1] * roi_scale_factor['y'])}")
    print(f"MSE(x) scaled {mse(y[:N, 21], y_qkeras[:N, 0])}")
    print(f"MSE(y) scaled {mse(y[:N, 12], y_qkeras[:N, 1])}")
    #print(f"MSE(cotB) {mse(y[:N, 12], y_qkeras[:N, 2])}")
else:
    print(f"MSE(x) {mse(y[:N, 0], y_qkeras[:N, 0])}")
    print(f"MSE(y) {mse(y[:N, 1], y_qkeras[:N, 1])}")
    #print(f"MSE(cotB) {mse(y[:N, 12], y_qkeras[:N, 2])}")

In [ ]:
DISTANCE_INFO_CSV_FILE = "/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/csv/dataset_2su_-9_9_48x192_50x12P5/dataset_2su_-9_9_48x192_50x12P5_test_t20_part_0_sum-optimized_compare-4n.csv"
ROI_INFO_CSV_FILE = "/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/tb/cocotb/cluster_id/dataset_2su_-9_9_48x192_50x12P5_csv/test/part.0.csv"
import pandas as pd
distance_info_df = pd.read_csv(DISTANCE_INFO_CSV_FILE, index_col="event-idx")
roi_info_df = pd.read_csv(ROI_INFO_CSV_FILE, index_col="event-idx")
display(roi_info_df)

In [ ]:
distance_roi_nn_merged = pd.merge(
    distance_info_df,
    roi_info_df,
    on='event-idx')

In [ ]:
display(distance_roi_nn_merged)

In [ ]:
cols = ["x-predicted", "y-predicted", "cotBeta-predicted"]
y_qkeras_df = pd.DataFrame(y_qkeras, columns=cols)
y_qkeras_df.index.name = "event-idx"

y_qkeras_df["x-predicted-scaled"]       = y_qkeras_df["x-predicted"]       * roi_scale_factor["x"]
y_qkeras_df["y-predicted-scaled"]       = y_qkeras_df["y-predicted"]       * roi_scale_factor["y"]
y_qkeras_df["cotBeta-predicted-scaled"] = y_qkeras_df["cotBeta-predicted"] * roi_scale_factor["cotB"]

In [ ]:
display(y_qkeras_df)

In [ ]:
distance_roi_nn_predictions_merged = pd.merge(
    distance_roi_nn_merged,
    y_qkeras_df,
    on='event-idx')
distance_roi_nn_predictions_merged.to_csv("distance_roi_nn_predictions.csv", index=False)

In [ ]:
display(distance_roi_nn_predictions_merged)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df = distance_roi_nn_predictions_merged  # keep original untouched

# pull columns
x_pred  = df["x-predicted-scaled"].to_numpy()
x_ref   = df["x-coi-roi-symmetric-physical"].to_numpy()
y_pred  = df["y-predicted-scaled"].to_numpy()
y_ref   = df["y-coi-roi-symmetric-physical"].to_numpy()
cotb_pred= df["cotBeta-predicted-scaled"].to_numpy()   # assumes you already created this
cotb_ref = df["cotBeta"].to_numpy()
dist    = df["distance"].to_numpy()

# residuals
x_err, y_err, cotb_err = x_pred - x_ref, y_pred - y_ref, cotb_pred - cotb_ref

# distance bins
binw   = 1.0
edges  = np.arange(np.floor(dist.min()), np.ceil(dist.max()) + binw, binw)
centers = (edges[:-1] + edges[1:]) / 2.0

def binned_mse(err, dist, edges):
    return np.array([
        np.nanmean((err[(dist >= lo) & (dist < hi)])**2) if np.any((dist >= lo) & (dist < hi)) else np.nan
        for lo, hi in zip(edges[:-1], edges[1:])
    ])

x_mse   = binned_mse(x_err,    dist, edges)
y_mse   = binned_mse(y_err,    dist, edges)
cotb_mse = binned_mse(cotb_err, dist, edges)

# one plot, three lines
plt.figure()
plt.plot(centers, x_mse,    marker="o", label="MSE(x)")
plt.plot(centers, y_mse,    marker="o", label="MSE(y)")
plt.plot(centers, cotb_mse, marker="o", label="MSE(cotBeta)")
plt.xlabel("distance (binned)")
plt.ylabel("MSE")
plt.title("MSE vs distance")
plt.grid(True)
plt.legend()
# plt.yscale("log")  # <- uncomment if ranges differ a lot


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_mse_vs_distance_separate(
    df,
    binw=1.0,
    x_pred_col="x-predicted-scaled",
    x_ref_col="x-coi-roi-symmetric-physical",
    y_pred_col="y-predicted-scaled",
    y_ref_col="y-coi-roi-symmetric-physical",
    cot_pred_col="cotBeta-predicted-scaled",  # "cotb" predicted
    cot_ref_col="cotBeta",                    # "cotb" reference
    dist_col="distance",
    ax_xy=None,
    ax_cot=None,
    logy=False,
    dist_min=None,
    dist_max=None,
    metric="MSE"
):
    # pull columns
    x_pred  = df[x_pred_col].to_numpy()
    x_ref   = df[x_ref_col].to_numpy()
    y_pred  = df[y_pred_col].to_numpy()
    y_ref   = df[y_ref_col].to_numpy()
    cot_pred= df[cot_pred_col].to_numpy()
    cot_ref = df[cot_ref_col].to_numpy()
    dist    = df[dist_col].to_numpy()

    # residuals
    x_err   = x_pred  - x_ref
    y_err   = y_pred  - y_ref
    cot_err = cot_pred- cot_ref

    # optional distance-range filter (exclude values outside range)
    mask = np.isfinite(dist)
    if dist_min is not None:
        mask &= (dist >= dist_min)
    if dist_max is not None:
        mask &= (dist <= dist_max)

    dist    = dist[mask]
    x_err   = x_err[mask]
    y_err   = y_err[mask]
    cotb_err = cot_err[mask]

    if dist.size == 0:
        raise ValueError("No samples within the requested distance range.")

    # distance bin edges & centers (based on filtered distances)
    edges   = np.arange(np.floor(np.nanmin(dist)), np.ceil(np.nanmax(dist)) + binw, binw)
    centers = (edges[:-1] + edges[1:]) / 2.0

    def agg_mae(e):    return np.nanmean(np.abs(e))
    def agg_mse(e):    return np.nanmean(np.square(e))
    def agg_rmse(e):   return np.sqrt(agg_mse(e))
    
    def binned_metric(err, dist, edges, agg):
        out = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (dist >= lo) & (dist < hi)
            out.append(agg(err[m]) if m.any() else np.nan)
        return np.array(out)

    if metric == "MAE":
        metric_func = agg_mae
    elif metric == "MSE":
        metric_func = agg_mse
    else:
        metric_func = agg_rmse
    
    x_metric   = binned_metric(x_err,   dist, edges, agg_mse)
    y_metric   = binned_metric(y_err,   dist, edges, agg_mse)
    cotb_metric = binned_metric(cotb_err, dist, edges, agg_mse)

    # plots
    if ax_xy is None:
        _, ax_xy = plt.subplots()
    ax_xy.plot(centers, x_metric,  marker="o", label=f"{metric}(x)")
    ax_xy.plot(centers, y_metric,  marker="o", label=f"{metric}(y)")
    ax_xy.set_xlabel("distance (binned)")
    ax_xy.set_ylabel(f"{metric}")
    ax_xy.set_title(f"X/Y {metric} vs distance")
    ax_xy.grid(True)
    ax_xy.legend()
    if logy:
        ax_xy.set_yscale("log")

    if ax_cot is None:
        _, ax_cot = plt.subplots()
    ax_cot.plot(centers, cotb_metric, marker="o", label=f"{metric}(cotBeta)")
    ax_cot.set_xlabel("distance (binned)")
    ax_cot.set_ylabel(f"{metric}")
    ax_cot.set_title(f"cotBeta {metric} vs distance")
    ax_cot.grid(True)
    ax_cot.legend()
    if logy:
        ax_cot.set_yscale("log")

    return centers, x_mse, y_mse, cot_mse, (ax_xy, ax_cot)



In [ ]:
_ = plot_mse_vs_distance_separate(distance_roi_nn_predictions_merged, metric="RMSE")

In [ ]:
_ = plot_mse_vs_distance_separate(distance_roi_nn_predictions_merged, dist_min=0, dist_max=8, metric="RMSE")